# DATA preparation

In [31]:
# !python.exe -m pip install --upgrade pip
#!pip install haystack-ai "transformers[torch,sentencepiece]"
# !pip install scrapeops-python-requests



In [32]:
# !pip install  langchain_community
# !python -m spacy download en_core_web_sm
# !pip install nltk spacy 
# !pip install beautifulsoup4
# !pip install Wikipedia-API
# !pip install googlesearch-python sentence_transformers pandas
# !pip install lxml

In [33]:
import os
import json
import numpy as np  
mushroomenvalv1 = "mushroom.en-tst.v1.jsonl"
mushroomenvalv1_output = "./mushroom.en-val.v1.output.json"

num_iteraciones = 2  # Adjust this number as needed
# Seed for reproducibility
seed_val = 442

# =========================
# Setup Environment
# =========================

# Create the output directory if it doesn't exist
output_dir = os.path.dirname(mushroomenvalv1_output)
os.makedirs(output_dir, exist_ok=True)

# =========================
# Load Data
# =========================

# Load the JSON data
data_val_all = []
with open(mushroomenvalv1, 'r', encoding='utf-8') as istr:
    for line in istr:
        try:
            data = json.loads(line.strip())
            if isinstance(data, dict) and 'id' in data:
                data_val_all.append(data)
        except json.JSONDecodeError:
            continue  # Skip invalid lines

num_sample = len(data_val_all)
num_iteraciones = 90  # Adjust this number as needed

print(f"Total de muestras en el conjunto: {len(data_val_all)}")

# Adjust the number of iterations to not exceed the total samples
# num_iteraciones = min(num_iteraciones, num_sample)
# num_iteraciones


data_val_all= data_val_all[130:]

Total de muestras en el conjunto: 154


# URL data extraction

In [34]:
query = []
response = []
ids = []
for model_input in data_val_all:
    query.append(model_input["model_input"])
    response.append(model_input["model_output_text"])
    ids.append(model_input["id"])
print(query)
print(response)
print(ids)

['Which Olympic sport did Gergely Kulcsár compete in?', 'Was there another Kalanga-speaking kingdom aside from the kingdom of Zimbabwe?', 'How many martellos are there on Formentera?', 'Is Yushu the largest city in Qinghai?', 'What was the population of Palm River-Clair Mel in 2010?', 'Explain in a few words who is Choe Ryong-hae.', 'What sport does Nina Curtis practice professionally?', 'Who developed International Klein Blue?', 'Who did Hugh Fraser play in Shapre?', 'Who plays Susan Delfino in Desperate Housewives?', 'Explain what was the Autonomous Province of Korçë in a few words.', 'When was Black Sabbath’s The Eternal Idol released?', 'In which country did Domantas Sabonis start his career?', 'In which French region is Saint-Voir located?', 'Did Health ever compose videogame soundtracks?', 'How many presidential candidates were there in the 2013 Iranian presidential election?', 'What is the perihelion of 661 Cloelia?', 'When did Karen Percy become a Member of the Order of Canada?

In [35]:


import pandas as pd

def create_dataframe(query, response, ids):
    """
    Create a DataFrame with query, response and URLs
    
    Args:
        query (str): The search query
        response (str): The response text
        urls (list): List of URLs from search results
    """
    # Create dictionary with the data
    data = {
        'query': query,
        'response': response,
        'id': ids
    }
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    
    return df

df = create_dataframe(query, response, ids)  # Note: query[0] since query is a tuple
df


,query,response,id
0,Which Olympic sport did Gergely Kulcsár compet...,Gerd Kulaşar competed in the men's hammer thr...,tst-en-131
1,Was there another Kalanga-speaking kingdom asi...,The Kalangas were not the only Kalango-speake...,tst-en-132
2,How many martellos are there on Formentera?,There are a total of 4 martello towers on the...,tst-en-133
3,Is Yushu the largest city in Qinghai?,"YUSHU, Qinghais second-largest city, located ...",tst-en-134
4,What was the population of Palm River-Clair Me...,"In 2010, Palm river-clair mel had a populatio...",tst-en-135
5,Explain in a few words who is Choe Ryong-hae.,Choi Young-ho was a North Korean politician w...,tst-en-136
6,What sport does Nina Curtis practice professio...,Nina is a professional surfer.\n,tst-en-137
7,Who developed International Klein Blue?,The dye was developed by a chemist named Ewal...,tst-en-138
8,Who did Hugh Fraser play in Shapre?,"Hugh ""Hughie"" Fraser Hughie Fraser (born Hugh...",tst-en-139
9,Who plays Susan Delfino in Desperate Housewives?,Susan is played by Felicity Huffman.\n,tst-en-140


# Mejorar la logica del retriver para obtener la información más relevante.

In [36]:
import urllib.parse

def create_google_search_link(query, max_pages=10):
  """
  Creates a Google Search URL for the given query.

  Args:
    query: The search query string.

  Returns:
    The Google Search URL.
  """
  encoded_query = urllib.parse.quote(query) 
  base_url = "https://www.google.com/search?"
  params = f"q={encoded_query}&num={max_pages}"
  return base_url + params

# Example usage
search_term = "how to bake a cake"
search_link = create_google_search_link(search_term)
print(search_link)

https://www.google.com/search?q=how%20to%20bake%20a%20cake&num=10


In [41]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd  # Assuming you are using a DataFrame to handle queries

# Headers to mimic browser requests
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
}

def google_search(query, desired_results=7, max_attempts=1):
    """
    Perform a Google search and extract results including snippets.
    Will retry searching with different result counts if initial attempts fail.
    
    Args:
        query (str): Search query
        desired_results (int): Number of results desired
        max_attempts (int): Maximum number of retry attempts
        
    Returns:
        list: List of search result dictionaries
    """
    search_results = []
    attempt = 0
    
    while len(search_results) < desired_results and attempt < max_attempts:
        # Increase results per attempt to improve chances of getting valid results
        num_results = desired_results * (attempt + 1)
        
        params = {
            'q': query,
            'gl': 'us',
            'num': num_results
        }
        
        try:
            # Make the request
            # response = requests.get(
            #     'https://www.google.com/search',
            #     headers=headers,
            #     params=params,
            #     timeout=30
            # )
            url = create_google_search_link(query)

            proxies = {
            "https": "scraperapi.render=true.country_code=xxx" ##este link se consigue des la interfaz de scraperapi
            }
            response = requests.get(url, proxies=proxies, verify=False)
            response.raise_for_status()
            
            # Parse the HTML
            soup = BeautifulSoup(response.text, 'lxml')
            
            # Extract results for this attempt
            for result in soup.select('div.g'):
                try:
                    title_element = result.select_one('h3')
                    title = title_element.text if title_element else None
                    
                    link_element = result.select_one('a')
                    link = link_element['href'] if link_element else None
                    
                    snippet_element = result.select_one('span.hgKElc') or result.select_one('.VwiC3b') or result.select_one('.kno-rdesc')
                    snippet = snippet_element.text if snippet_element else None
                    
                    if title or snippet:  # Only add if we have at least title or snippet
                        search_results.append({
                            'title': title,
                            'link': link,
                            'snippet': snippet,
                        })
                except Exception as e:
                    print(f"Error parsing result: {str(e)}")
                    continue
            
            # If we got some results, wait before next attempt
            if search_results:
                time.sleep(3)
                
        except requests.RequestException as e:
            print(f"Request error on attempt {attempt + 1}: {str(e)}")
        except Exception as e:
            print(f"Unexpected error on attempt {attempt + 1}: {str(e)}")
        attempt += 1
    return search_results

df["snippes_google"] = df["query"].apply(google_search)
df["snippes_google"] 

/workspace/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy-server.scraperapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


KeyboardInterrupt: 

In [98]:
def fill_missing(row):
    list_snippets_google = row["snippes_google_v2"]
    if not list_snippets_google or list_snippets_google == 0:  # Handle empty/missing cases
        return google_search(row["query"])
    return list_snippets_google  # Return original value if not empty

df["snippes_google_v2"] = df.apply(fill_missing, axis=1)

/workspace/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy-server.scraperapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/workspace/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy-server.scraperapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [99]:
df["snippes_google_v2"]

0     [{'title': 'Javelin throw - Wikipedia', 'link'...
1     [{'title': 'Kalanga people', 'link': 'https://...
2     [{'title': 'Formentera', 'link': 'https://en.w...
3     [{'title': 'Yushu City, Qinghai - Wikipedia', ...
4     [{'title': 'Palm River-Clair Mel CDP ... - U.S...
5     [{'title': 'Choe Ryong-hae - Wikipedia', 'link...
6     [{'title': 'Nina CURTIS', 'link': 'https://oly...
7     [{'title': 'International Klein Blue - Wikiped...
8     [{'title': 'Hugh Fraser (actor)', 'link': 'htt...
9     [{'title': 'Susan Delfino | Wiksteria Lane - F...
10    [{'title': 'Autonomous Province of Korçë - Wik...
11    [{'title': 'The Eternal Idol', 'link': 'https:...
12    [{'title': 'Gonzaga Bulldogs men's basketball ...
13    [{'title': 'Saint-Voir - Wikipedia', 'link': '...
14    [{'title': 'Health (band) - Wikipedia', 'link'...
15    [{'title': '2013 Iranian presidential election...
16    [{'title': '661 Cloelia - Wikipedia', 'link': ...
17    [{'title': 'Karen Percy - Team Canada - Ca

In [94]:
print(df["query"][0])
df["snippes_google"][0]

Which Olympic sport did Gergely Kulcsár compete in?


[]

In [100]:
def fix_dict_to_list(list_of_dicts):
    lista=[]
    for dicts in list_of_dicts:
        if dicts["snippet"]:
            lista.append(dicts["snippet"])
    return lista
df["scraping_and_procesor"] = df["snippes_google"].apply(fix_dict_to_list)

In [102]:
df.to_csv("scraping_and_procesor3.csv", index=False)
